In [ ]:
# Change this to your preferred framework (e.g., 'cuda', 'pytorch', 'triton', 'jax', 'mojo')
EVAL_LANG = 'cuda'

SAVE_GPU = True


<p>
    Implement a basic General Matrix Multiplication (GEMM). Given matrix $A$ of dimensions $M \times K$, matrix $B$ of dimensions $K \times N$, input/output matrix $C$ of dimensions $M \times N$, and scalar multipliers $ \alpha $ and $ \beta $, compute the operation:
    $$ C = \alpha \cdot (A \times B) + \beta \cdot C_{initial} $$
</p>
<p>
    The input matrices $A$, $B$, and the initial state of $C$ contain 16-bit floating-point numbers (FP16/<code>half</code>). All matrices are stored in row-major order. The scalars $ \alpha $ and $ \beta $ are 32-bit floats.
</p>

<h2>Implementation Requirements</h2>
<ul>
    <li>Use only native features (external libraries other than WMMA are not permitted).</li>
    <li>The <code>solve</code> function signature must remain unchanged.</li>
    <li>Accumulation during multiplication should use FP32 for better precision before converting the final result to FP16.</li>
    <li>The final result must be stored back into matrix <code>C</code> as <code>half</code>.</li>
</ul>

<h2>Example:</h2>
<p>
Input:<br>
<em>(Note: Input matrices A, B, C_initial are FP16 type for the problem)</em><br>
Matrix $A$ ($M=2, K=3$):
$$
\begin{bmatrix}
1.0 & 2.0 & 3.0 \\
4.0 & 5.0 & 6.0
\end{bmatrix}
$$
Matrix $B$ ($K=3, N=2$):
$$
\begin{bmatrix}
1.0 & 2.0 \\
3.0 & 4.0 \\
5.0 & 6.0
\end{bmatrix}
$$
Matrix $C_{initial}$ ($M=2, N=2$):
$$
\begin{bmatrix}
1.0 & 1.0 \\
1.0 & 1.0
\end{bmatrix}
$$
$$\alpha = 1.0 \text{ (FP32)}$$
$$\beta = 0.0 \text{ (FP32)}$$

Output (FP16):<br>
Matrix $C$ ($M=2, N=2$):
$$
\begin{bmatrix}
22.0 & 28.0 \\
49.0 & 64.0
\end{bmatrix}
$$
</p>

<h2>Constraints</h2>
<ul>
    <li>16 &le; <code>M</code>, <code>N</code>, <code>K</code> &le; 4096</li>

  <li>Performance is measured with <code>K</code> = 1,024, <code>M</code> = 1,024, <code>N</code> = 1,024</li>
</ul>


# CUDA

In [ ]:
%%writefile solution.cu
#include <cuda_fp16.h>
#include <cuda_runtime.h>

// A, B, and C are device pointers
extern "C" void solve(const half* A, const half* B, half* C, int M, int N, int K, float alpha,
                      float beta) {}


# CUTE

In [ ]:
%%writefile solution_cute.py
import cutlass
import cutlass.cute as cute


# A, B, C are tensors on the GPU
@cute.jit
def solve(
    A: cute.Tensor,
    B: cute.Tensor,
    C: cute.Tensor,
    M: cute.Int32,
    N: cute.Int32,
    K: cute.Int32,
    alpha: cute.Float32,
    beta: cute.Float32,
):
    pass


# JAX

In [ ]:
%%writefile solution_jax.py
import jax
import jax.numpy as jnp


# A, B are tensors on the GPU
@jax.jit
def solve(
    A: jax.Array, B: jax.Array, M: int, N: int, K: int, alpha: float, beta: float
) -> jax.Array:
    # return output tensor directly
    pass


# MOJO

In [ ]:
%%writefile solution.mojo
from std.gpu.host import DeviceContext
from std.gpu import block_dim, block_idx, thread_idx
from std.memory import UnsafePointer
from std.math import ceildiv


@export
def solve(
    A: UnsafePointer[Float16, MutExternalOrigin],
    B: UnsafePointer[Float16, MutExternalOrigin],
    C: UnsafePointer[Float16, MutExternalOrigin],
    M: Int32,
    N: Int32,
    K: Int32,
    alpha: Float32,
    beta: Float32,
) raises:
    pass


# Torch

In [ ]:
%%writefile solution_pytorch.py
import torch


# A, B, C are tensors on the GPU
def solve(
    A: torch.Tensor,
    B: torch.Tensor,
    C: torch.Tensor,
    M: int,
    N: int,
    K: int,
    alpha: float,
    beta: float,
):
    pass


# Triton

In [ ]:
%%writefile solution_triton.py
import torch
import triton
import triton.language as tl


# a, b, c are tensors on the GPU
def solve(
    a: torch.Tensor,
    b: torch.Tensor,
    c: torch.Tensor,
    M: int,
    N: int,
    K: int,
    alpha: float,
    beta: float,
):
    pass


# Evaluate Setup

In [ ]:
# Download required files from GitHub
!mkdir -p core
!wget -q https://raw.githubusercontent.com/lekhit/leetgpu-challenges/main/challenges/core/challenge_base.py -O core/challenge_base.py
!wget -q https://raw.githubusercontent.com/lekhit/leetgpu-challenges/main/challenges/core/evaluator.py -O core/evaluator.py
!wget -q https://raw.githubusercontent.com/lekhit/leetgpu-challenges/main/challenges/medium/22_gemm/challenge.py -O challenge.py

from challenge import Challenge
from core.evaluator import Evaluate

ch = Challenge()


# Evaluation code

In [ ]:
# Run the evaluator based on configuration
if EVAL_LANG == 'cuda':
    Evaluate.eval_cuda(ch)
elif EVAL_LANG in ['pytorch', 'triton', 'jax', 'cute']:
    Evaluate.eval_python(ch, EVAL_LANG)
elif EVAL_LANG == 'mojo':
    Evaluate.eval_mojo(ch)
else:
    print(f"Unknown language {EVAL_LANG}")

# Disconnect runtime to save Colab resources
if SAVE_GPU:
    from google.colab import runtime
    runtime.unassign()
